In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error

Load the Dataset

In [2]:
df = pd.read_csv("data/taxi_pickups_area.csv")
df.shape

(8064, 79)

Convert Trip Start Timestamp to datetime object

In [3]:
areas = [c for c in df.columns.to_list() if c != 'Trip Start Timestamp']
df['Trip Start Timestamp'] = pd.to_datetime(df['Trip Start Timestamp'])

Train validation split

In [4]:
train_size = int(len(df) * 0.8)

train = df[:train_size]
val = df[train_size:]

train.shape

(6451, 79)

In [5]:
val.shape

(1613, 79)

Time Series Preprocessing

Check Missing values and time frequency

In [6]:
# missing = train[train.isnan()].count()
train[train.isna()].count()

Trip Start Timestamp        0
Pickup Community Area_0     0
Pickup Community Area_1     0
Pickup Community Area_2     0
Pickup Community Area_3     0
                           ..
Pickup Community Area_73    0
Pickup Community Area_74    0
Pickup Community Area_75    0
Pickup Community Area_76    0
Pickup Community Area_77    0
Length: 79, dtype: int64

no missing values detected

Check time Frequency

In [7]:
pd.infer_freq(train[train.columns[0]])

'15min'

Time Frequency constant 15-min interval

Handle anomalies with STL Decomposition for each area

In [27]:
from statsmodels.tsa.seasonal import STL

cleaned_train = train.copy()

# def detect_anomalies(serie, robust=True, period=672, threshold=3.0): # an be used!

#     stl = STL(serie, robust=robust, period=period)
#     result = stl.fit()
#     resid = result.resid

#     # Robust center/scale — mean/std get skewed by the very anomalies we want to find
#     median = resid.median()
#     mad = np.median(np.abs(resid - median))
#     mad_std = mad * 1.4826 if mad != 0 else resid.std()  # fallback if MAD is 0

#     lower = median - (threshold * mad_std)
#     upper = median + (threshold * mad_std)
#     return serie[(resid < lower) | (resid > upper)]

def detect_anomalies(serie, robust=True, period=672):

    stl = STL(serie, robust=robust, period=period)

    result = stl.fit()

    resid =  result.resid

    resid_m = resid.mean()
    resid_dv = resid.std()

    lower = resid_m - (3 * resid_dv) 
    upper = resid_m + (3 * resid_dv)

    return serie[(resid < lower) | (resid > upper)]


for area in areas:
    
    serie = train[area]

    anomalies = detect_anomalies(serie)
    
    serie.loc[anomalies.index] = np.nan

    serie = serie.interpolate(
        method="linear",
        limit_direction="both"
    )

    cleaned_train[area] = serie


save cleaned train time series

In [28]:
import joblib
joblib.dump(cleaned_train, "cleaned_train.pkl")

['cleaned_train.pkl']

Load Cleaned train set

In [5]:
import joblib

cleaned_train = joblib.load("cleaned_train.pkl")
# print(train["Pickup Community Area_25"].hasnans)

In [ ]:
Log Transformation

In [6]:
transformed_train = cleaned_train

for area in areas:
    transformed_train[area] = np.log1p(transformed_train[area])

Check and make time series stationary for each Area with Augmented Dickey fuller test and differencing

In [8]:
from statsmodels.tsa.stattools import adfuller
# import matplotlib.pyplot as plt


def check_stationarity(serie, significance_level=0.05):

    # Drop missing values caused by differencing
    # clean_serie = pd.Series(serie).dropna()
    
    # Handle edge case: empty series or zero variance
    # if len(clserie) < 10 or serie.nunique() <= 1:
    #     return False

    result = adfuller(serie, autolag='AIC')
    p_value = result[1]

    return p_value < significance_level


def make_stationary_diff(serie, max_diff=3, significance_level=0.05):

    current_serie = pd.Series(serie).copy()
    diff_count = 0

    while diff_count < max_diff:
        if check_stationarity(current_serie, significance_level):
            break
            
        current_serie = current_serie.diff().dropna()
        diff_count += 1

    return current_serie, diff_count


for area in areas:
    serie, diff_count = make_stationary_diff(transformed_train[area])
    transformed_train[area] = serie

transformed_train.isna().sum()

ValueError: Invalid input, x is constant

Fit Naive averages using Global average for each Area

In [10]:
global_averages = transformed_train[areas].mean()
global_averages

Pickup Community Area_0     1.798926
Pickup Community Area_1     0.628776
Pickup Community Area_2     0.616233
Pickup Community Area_3     1.134126
Pickup Community Area_4     0.529569
                              ...   
Pickup Community Area_73    0.068246
Pickup Community Area_74    0.000000
Pickup Community Area_75    0.030954
Pickup Community Area_76    2.921165
Pickup Community Area_77    0.991413
Length: 78, dtype: float64

Forecast each Area

In [11]:
forecast = pd.Series(index=areas)
for area in areas:
    forecast[area] = np.expm1(global_averages[area])

forecast

Pickup Community Area_0      5.043151
Pickup Community Area_1      0.875314
Pickup Community Area_2      0.851938
Pickup Community Area_3      2.108455
Pickup Community Area_4      0.698201
                              ...    
Pickup Community Area_73     0.070629
Pickup Community Area_74     0.000000
Pickup Community Area_75     0.031438
Pickup Community Area_76    17.562896
Pickup Community Area_77     1.695041
Length: 78, dtype: float64

Compute Mean Absolute Error using validation

In [12]:

all_mea = []
for area in areas:
    area_pred = np.ones(len(val)) * forecast[area]
    mae = mean_absolute_error(area_pred, val[area])
    all_mea.append(mae)

average_mae = np.array(all_mea).mean()
print(f"average of all area mean absolute error : {average_mae}")

average of all area mean absolute error : 1.3989638609718165


Load Taxi submission Dataset

In [19]:
sub_df = pd.read_csv("data/taxi_submission_file.csv")

Forecast with naive averages and save

In [14]:
sub_pred = pd.DataFrame(columns=sub_df.columns)
sub_pred['Trip Start Timestamp'] = sub_df['Trip Start Timestamp']

for area in areas:
    area_pred = np.ones(len(sub_df)) * forecast[area]
    sub_pred[area] = np.floor(area_pred)


sub_pred

,Trip Start Timestamp,Pickup Community Area_0,Pickup Community Area_1,Pickup Community Area_2,Pickup Community Area_3,Pickup Community Area_4,Pickup Community Area_5,Pickup Community Area_6,Pickup Community Area_7,Pickup Community Area_8,...,Pickup Community Area_68,Pickup Community Area_69,Pickup Community Area_70,Pickup Community Area_71,Pickup Community Area_72,Pickup Community Area_73,Pickup Community Area_74,Pickup Community Area_75,Pickup Community Area_76,Pickup Community Area_77
0,2019-06-24 00:00:00,5.0,0.0,0.0,2.0,0.0,0.0,7.0,4.0,55.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.0,1.0
1,2019-06-24 00:15:00,5.0,0.0,0.0,2.0,0.0,0.0,7.0,4.0,55.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.0,1.0
2,2019-06-24 00:30:00,5.0,0.0,0.0,2.0,0.0,0.0,7.0,4.0,55.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.0,1.0
3,2019-06-24 00:45:00,5.0,0.0,0.0,2.0,0.0,0.0,7.0,4.0,55.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.0,1.0
4,2019-06-24 01:00:00,5.0,0.0,0.0,2.0,0.0,0.0,7.0,4.0,55.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667,2019-06-30 22:45:00,5.0,0.0,0.0,2.0,0.0,0.0,7.0,4.0,55.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.0,1.0
668,2019-06-30 23:00:00,5.0,0.0,0.0,2.0,0.0,0.0,7.0,4.0,55.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.0,1.0
669,2019-06-30 23:15:00,5.0,0.0,0.0,2.0,0.0,0.0,7.0,4.0,55.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.0,1.0
670,2019-06-30 23:30:00,5.0,0.0,0.0,2.0,0.0,0.0,7.0,4.0,55.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.0,1.0


Fit Simple Moving Averages

In [15]:
window = 10
sma_mae = []
sma_series = pd.Series(index=areas)
for area in areas:

    avg_window = transformed_train[area].tail(window).mean() # Fit using scaled values


    inversed_avg = np.expm1(avg_window)
    sma_series[area] = inversed_avg # hold the inversed forecast for each area
    forecast = np.ones(len(val)) * inversed_avg # just create val size of the sca
    sma_mae.append(mean_absolute_error(forecast, val[area]))

average_mae = np.array(sma_mae).mean()
print(f"average mean absolute error :{average_mae}")

average mean absolute error :0.8337175520198017


In [16]:
sub_pred = pd.DataFrame(columns=sub_df.columns)
sub_pred['Trip Start Timestamp'] = sub_df['Trip Start Timestamp']

for area in areas:
    area_pred = np.ones(len(sub_df)) * sma_series[area]
    sub_pred[area] = area_pred

sub_pred

,Trip Start Timestamp,Pickup Community Area_0,Pickup Community Area_1,Pickup Community Area_2,Pickup Community Area_3,Pickup Community Area_4,Pickup Community Area_5,Pickup Community Area_6,Pickup Community Area_7,Pickup Community Area_8,...,Pickup Community Area_68,Pickup Community Area_69,Pickup Community Area_70,Pickup Community Area_71,Pickup Community Area_72,Pickup Community Area_73,Pickup Community Area_74,Pickup Community Area_75,Pickup Community Area_76,Pickup Community Area_77
0,2019-06-24 00:00:00,0.966307,0.196231,0.0,0.390389,0.071773,0.0,0.578437,0.597138,4.983121,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.578437,0.148698
1,2019-06-24 00:15:00,0.966307,0.196231,0.0,0.390389,0.071773,0.0,0.578437,0.597138,4.983121,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.578437,0.148698
2,2019-06-24 00:30:00,0.966307,0.196231,0.0,0.390389,0.071773,0.0,0.578437,0.597138,4.983121,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.578437,0.148698
3,2019-06-24 00:45:00,0.966307,0.196231,0.0,0.390389,0.071773,0.0,0.578437,0.597138,4.983121,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.578437,0.148698
4,2019-06-24 01:00:00,0.966307,0.196231,0.0,0.390389,0.071773,0.0,0.578437,0.597138,4.983121,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.578437,0.148698
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667,2019-06-30 22:45:00,0.966307,0.196231,0.0,0.390389,0.071773,0.0,0.578437,0.597138,4.983121,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.578437,0.148698
668,2019-06-30 23:00:00,0.966307,0.196231,0.0,0.390389,0.071773,0.0,0.578437,0.597138,4.983121,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.578437,0.148698
669,2019-06-30 23:15:00,0.966307,0.196231,0.0,0.390389,0.071773,0.0,0.578437,0.597138,4.983121,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.578437,0.148698
670,2019-06-30 23:30:00,0.966307,0.196231,0.0,0.390389,0.071773,0.0,0.578437,0.597138,4.983121,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.578437,0.148698


Fit Moving Averages using different window slice

In [17]:

window = 48
sma_mae = []
sma_series = pd.Series(index=areas)
for area in areas:

    avg_window = transformed_train[area].tail(window).mean() # Fit using scaled values


    inversed_avg = np.expm1(avg_window)
    sma_series[area] = inversed_avg # hold the inversed forecast for each area
    forecast = np.ones(len(val)) * inversed_avg # just create val size of the sca
    sma_mae.append(mean_absolute_error(forecast, val[area]))

average_mae = np.array(sma_mae).mean()
print(f"average mean absolute error :{average_mae}")

average mean absolute error :0.630961661042094


In [18]:
sub_pred = pd.DataFrame(columns=sub_df.columns)
sub_pred['Trip Start Timestamp'] = sub_df['Trip Start Timestamp']

for area in areas:
    area_pred = np.ones(len(sub_df)) * sma_series[area]
    sub_pred[area] = np.floor(area_pred)

sub_pred

,Trip Start Timestamp,Pickup Community Area_0,Pickup Community Area_1,Pickup Community Area_2,Pickup Community Area_3,Pickup Community Area_4,Pickup Community Area_5,Pickup Community Area_6,Pickup Community Area_7,Pickup Community Area_8,...,Pickup Community Area_68,Pickup Community Area_69,Pickup Community Area_70,Pickup Community Area_71,Pickup Community Area_72,Pickup Community Area_73,Pickup Community Area_74,Pickup Community Area_75,Pickup Community Area_76,Pickup Community Area_77
0,2019-06-24 00:00:00,4.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0
1,2019-06-24 00:15:00,4.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0
2,2019-06-24 00:30:00,4.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0
3,2019-06-24 00:45:00,4.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0
4,2019-06-24 01:00:00,4.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667,2019-06-30 22:45:00,4.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0
668,2019-06-30 23:00:00,4.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0
669,2019-06-30 23:15:00,4.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0
670,2019-06-30 23:30:00,4.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,17.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.0,0.0


Fit Weighted Moving Averages

In [19]:
# transformed_train = train.copy()

window = 10
weights = np.arange(1, window + 1)

sma_mae = []
sma_series = pd.Series(index=areas)
for area in areas:

    values = transformed_train[area].tail(window)
    # avg_window = (transformed_train[area].tail(window) * weights).mean() # Fit using scaled values

    avg_window = np.average(values, weights=weights)
    inversed_avg = np.expm1(avg_window)
    sma_series[area] = inversed_avg # hold the inversed forecast for each area
    forecast = np.ones(len(val)) * inversed_avg # just create val size of the sca
    sma_mae.append(mean_absolute_error(forecast, val[area]))


average_mae = np.array(sma_mae).mean()
print(f"average mean absolute error :{average_mae}")

average mean absolute error :0.839195618656573


In [20]:
sub_pred = pd.DataFrame(columns=sub_df.columns)
sub_pred['Trip Start Timestamp'] = sub_df['Trip Start Timestamp']

for area in areas:
    area_pred = np.ones(len(sub_df)) * sma_series[area]
    sub_pred[area] = area_pred

sub_pred

,Trip Start Timestamp,Pickup Community Area_0,Pickup Community Area_1,Pickup Community Area_2,Pickup Community Area_3,Pickup Community Area_4,Pickup Community Area_5,Pickup Community Area_6,Pickup Community Area_7,Pickup Community Area_8,...,Pickup Community Area_68,Pickup Community Area_69,Pickup Community Area_70,Pickup Community Area_71,Pickup Community Area_72,Pickup Community Area_73,Pickup Community Area_74,Pickup Community Area_75,Pickup Community Area_76,Pickup Community Area_77
0,2019-06-24 00:00:00,1.387096,0.206945,0.0,0.270864,0.078548,0.0,0.595988,0.461224,4.304777,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.400996,0.163267
1,2019-06-24 00:15:00,1.387096,0.206945,0.0,0.270864,0.078548,0.0,0.595988,0.461224,4.304777,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.400996,0.163267
2,2019-06-24 00:30:00,1.387096,0.206945,0.0,0.270864,0.078548,0.0,0.595988,0.461224,4.304777,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.400996,0.163267
3,2019-06-24 00:45:00,1.387096,0.206945,0.0,0.270864,0.078548,0.0,0.595988,0.461224,4.304777,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.400996,0.163267
4,2019-06-24 01:00:00,1.387096,0.206945,0.0,0.270864,0.078548,0.0,0.595988,0.461224,4.304777,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.400996,0.163267
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667,2019-06-30 22:45:00,1.387096,0.206945,0.0,0.270864,0.078548,0.0,0.595988,0.461224,4.304777,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.400996,0.163267
668,2019-06-30 23:00:00,1.387096,0.206945,0.0,0.270864,0.078548,0.0,0.595988,0.461224,4.304777,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.400996,0.163267
669,2019-06-30 23:15:00,1.387096,0.206945,0.0,0.270864,0.078548,0.0,0.595988,0.461224,4.304777,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.400996,0.163267
670,2019-06-30 23:30:00,1.387096,0.206945,0.0,0.270864,0.078548,0.0,0.595988,0.461224,4.304777,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.400996,0.163267


Simple Exponential Smoothing (SES)

Transform validation set

In [29]:
transformed_val = val.copy()
for area in areas:
    transformed_val[area] = np.log1p(transformed_val[area])

In [43]:
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

sma_mae = []
val_models = {}   # models fit on train only (for validation scoring)
final_models = {} # models refit on train+val (for submission)


for area in areas:
    # --- Step 1: fit on train, validate ---
    y_train = transformed_train[area] + np.random.normal(0, 1e-8, size=len(transformed_train[area]))
    
    model = SimpleExpSmoothing(y_train, initialization_method="estimated")
    fit_model = model.fit(optimized=True)
    val_models[area] = fit_model

    val_forecast = fit_model.forecast(steps=len(val))
    val_forecast = np.expm1(val_forecast)
    sma_mae.append(mean_absolute_error(val[area], val_forecast))

    # --- Step 2: refit on train+val for the actual test/submission forecast ---
    y_full = pd.concat([transformed_train[area], val[area]])  # or however val is stored in log/transformed space
    y_full = y_full + np.random.normal(0, 1e-8, size=len(y_full))

    final_model = SimpleExpSmoothing(y_full, initialization_method="estimated")
    final_model = final_model.fit(optimized=True)
    final_models[area] = final_model

print("Average validation MAE:", np.mean(sma_mae))

Average validation MAE: 0.7571781430922346


Submission with Simple Exponential Model

In [46]:
sub_pred = pd.DataFrame(columns=sub_df.columns)
sub_pred['Trip Start Timestamp'] = sub_df['Trip Start Timestamp']

for area in areas:
    forecast = final_models[area].forecast(steps=len(sub_df))
    forecast = forecast
    sub_pred[area] = forecast.to_numpy()

sub_pred

,Trip Start Timestamp,Pickup Community Area_0,Pickup Community Area_1,Pickup Community Area_2,Pickup Community Area_3,Pickup Community Area_4,Pickup Community Area_5,Pickup Community Area_6,Pickup Community Area_7,Pickup Community Area_8,...,Pickup Community Area_68,Pickup Community Area_69,Pickup Community Area_70,Pickup Community Area_71,Pickup Community Area_72,Pickup Community Area_73,Pickup Community Area_74,Pickup Community Area_75,Pickup Community Area_76,Pickup Community Area_77
0,2019-06-24 00:00:00,3.897937,0.502423,0.257196,0.292226,0.004102,0.087128,2.790172,0.605197,6.822294,...,0.025825,0.086172,0.000026,0.014917,0.003017,0.006723,0.00129,0.009731,15.289062,0.090059
1,2019-06-24 00:15:00,3.897937,0.502423,0.257196,0.292226,0.004102,0.087128,2.790172,0.605197,6.822294,...,0.025825,0.086172,0.000026,0.014917,0.003017,0.006723,0.00129,0.009731,15.289062,0.090059
2,2019-06-24 00:30:00,3.897937,0.502423,0.257196,0.292226,0.004102,0.087128,2.790172,0.605197,6.822294,...,0.025825,0.086172,0.000026,0.014917,0.003017,0.006723,0.00129,0.009731,15.289062,0.090059
3,2019-06-24 00:45:00,3.897937,0.502423,0.257196,0.292226,0.004102,0.087128,2.790172,0.605197,6.822294,...,0.025825,0.086172,0.000026,0.014917,0.003017,0.006723,0.00129,0.009731,15.289062,0.090059
4,2019-06-24 01:00:00,3.897937,0.502423,0.257196,0.292226,0.004102,0.087128,2.790172,0.605197,6.822294,...,0.025825,0.086172,0.000026,0.014917,0.003017,0.006723,0.00129,0.009731,15.289062,0.090059
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667,2019-06-30 22:45:00,3.897937,0.502423,0.257196,0.292226,0.004102,0.087128,2.790172,0.605197,6.822294,...,0.025825,0.086172,0.000026,0.014917,0.003017,0.006723,0.00129,0.009731,15.289062,0.090059
668,2019-06-30 23:00:00,3.897937,0.502423,0.257196,0.292226,0.004102,0.087128,2.790172,0.605197,6.822294,...,0.025825,0.086172,0.000026,0.014917,0.003017,0.006723,0.00129,0.009731,15.289062,0.090059
669,2019-06-30 23:15:00,3.897937,0.502423,0.257196,0.292226,0.004102,0.087128,2.790172,0.605197,6.822294,...,0.025825,0.086172,0.000026,0.014917,0.003017,0.006723,0.00129,0.009731,15.289062,0.090059
670,2019-06-30 23:30:00,3.897937,0.502423,0.257196,0.292226,0.004102,0.087128,2.790172,0.605197,6.822294,...,0.025825,0.086172,0.000026,0.014917,0.003017,0.006723,0.00129,0.009731,15.289062,0.090059


Double Exponential Smoothing (Holt's Linear Trend)

In [47]:
from statsmodels.tsa.holtwinters import Holt

sma_mae = []
val_models = {}   # models fit on train only (for validation scoring)
final_models = {} # models refit on train+val (for submission)


for area in areas:
    # --- Step 1: fit on train, validate ---
    y_train = transformed_train[area] + np.random.normal(0, 1e-8, size=len(transformed_train[area]))
    
    model = Holt(y_train, initialization_method="estimated")
    fit_model = model.fit(optimized=True)
    val_models[area] = fit_model

    val_forecast = fit_model.forecast(steps=len(val))
    val_forecast = np.expm1(val_forecast)
    sma_mae.append(mean_absolute_error(val[area], val_forecast))

    # --- Step 2: refit on train+val for the actual test/submission forecast ---
    y_full = pd.concat([transformed_train[area], val[area]])  # or however val is stored in log/transformed space
    y_full = y_full + np.random.normal(0, 1e-8, size=len(y_full))

    final_model = SimpleExpSmoothing(y_full, initialization_method="estimated")
    final_model = final_model.fit(optimized=True)
    final_models[area] = final_model

print("Average validation MAE:", np.mean(sma_mae))

Average validation MAE: 1.7856334186237767e+38


Submission with Holt Model

In [ ]:
sub_pred = pd.DataFrame(columns=sub_df.columns)
sub_pred['Trip Start Timestamp'] = sub_df['Trip Start Timestamp']

for area in areas:
    forecast = final_models[area].forecast(steps=len(sub_df))
    forecast = forecast
    sub_pred[area] = forecast.to_numpy()

sub_pred

In [ ]:
Triple Exponential Smoothing (Holt-Winters)

In [49]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

sma_mae = []
val_models = {}   # models fit on train only (for validation scoring)
final_models = {} # models refit on train+val (for submission)


for area in areas:
    # --- Step 1: fit on train, validate ---
    y_train = transformed_train[area] + np.random.normal(0, 1e-8, size=len(transformed_train[area]))
    
    model = ExponentialSmoothing(y_train, initialization_method="estimated")
    fit_model = model.fit(optimized=True)
    val_models[area] = fit_model

    val_forecast = fit_model.forecast(steps=len(val))
    val_forecast = np.expm1(val_forecast)
    sma_mae.append(mean_absolute_error(val[area], val_forecast))

    # --- Step 2: refit on train+val for the actual test/submission forecast ---
    y_full = pd.concat([transformed_train[area], val[area]])  # or however val is stored in log/transformed space
    y_full = y_full + np.random.normal(0, 1e-8, size=len(y_full))

    final_model = SimpleExpSmoothing(y_full, initialization_method="estimated")
    final_model = final_model.fit(optimized=True)
    final_models[area] = final_model

print("Average validation MAE:", np.mean(sma_mae))

Average validation MAE: 0.7571781358048353


Submisstion with Holt-winters

In [50]:
sub_pred = pd.DataFrame(columns=sub_df.columns)
sub_pred['Trip Start Timestamp'] = sub_df['Trip Start Timestamp']

for area in areas:
    forecast = final_models[area].forecast(steps=len(sub_df))
    forecast = forecast
    sub_pred[area] = forecast.to_numpy()

sub_pred

,Trip Start Timestamp,Pickup Community Area_0,Pickup Community Area_1,Pickup Community Area_2,Pickup Community Area_3,Pickup Community Area_4,Pickup Community Area_5,Pickup Community Area_6,Pickup Community Area_7,Pickup Community Area_8,...,Pickup Community Area_68,Pickup Community Area_69,Pickup Community Area_70,Pickup Community Area_71,Pickup Community Area_72,Pickup Community Area_73,Pickup Community Area_74,Pickup Community Area_75,Pickup Community Area_76,Pickup Community Area_77
0,2019-06-24 00:00:00,3.897937,0.502415,0.257196,0.292226,0.004102,0.087128,2.790172,0.605197,6.822294,...,0.025825,0.086172,0.000026,0.014917,0.003017,0.006723,0.00129,0.009731,15.289063,0.090059
1,2019-06-24 00:15:00,3.897937,0.502415,0.257196,0.292226,0.004102,0.087128,2.790172,0.605197,6.822294,...,0.025825,0.086172,0.000026,0.014917,0.003017,0.006723,0.00129,0.009731,15.289063,0.090059
2,2019-06-24 00:30:00,3.897937,0.502415,0.257196,0.292226,0.004102,0.087128,2.790172,0.605197,6.822294,...,0.025825,0.086172,0.000026,0.014917,0.003017,0.006723,0.00129,0.009731,15.289063,0.090059
3,2019-06-24 00:45:00,3.897937,0.502415,0.257196,0.292226,0.004102,0.087128,2.790172,0.605197,6.822294,...,0.025825,0.086172,0.000026,0.014917,0.003017,0.006723,0.00129,0.009731,15.289063,0.090059
4,2019-06-24 01:00:00,3.897937,0.502415,0.257196,0.292226,0.004102,0.087128,2.790172,0.605197,6.822294,...,0.025825,0.086172,0.000026,0.014917,0.003017,0.006723,0.00129,0.009731,15.289063,0.090059
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667,2019-06-30 22:45:00,3.897937,0.502415,0.257196,0.292226,0.004102,0.087128,2.790172,0.605197,6.822294,...,0.025825,0.086172,0.000026,0.014917,0.003017,0.006723,0.00129,0.009731,15.289063,0.090059
668,2019-06-30 23:00:00,3.897937,0.502415,0.257196,0.292226,0.004102,0.087128,2.790172,0.605197,6.822294,...,0.025825,0.086172,0.000026,0.014917,0.003017,0.006723,0.00129,0.009731,15.289063,0.090059
669,2019-06-30 23:15:00,3.897937,0.502415,0.257196,0.292226,0.004102,0.087128,2.790172,0.605197,6.822294,...,0.025825,0.086172,0.000026,0.014917,0.003017,0.006723,0.00129,0.009731,15.289063,0.090059
670,2019-06-30 23:30:00,3.897937,0.502415,0.257196,0.292226,0.004102,0.087128,2.790172,0.605197,6.822294,...,0.025825,0.086172,0.000026,0.014917,0.003017,0.006723,0.00129,0.009731,15.289063,0.090059


Fit and Select Best SARIMA models Based on AIC

In [15]:
import pmdarima as pm

sarima_models = {}

for area in areas:
    print(f"Fit SARIMA model for {area}...")
    model = pm.auto_arima(
        transformed_train[area],
        seasonal=False,
    
        start_p=0,
        start_q=0,
        max_p=3,
        max_q=3,
    
        d=None,
        max_d=2,
    
        information_criterion="aic",
    
        stepwise=False,
        n_jobs=-1,
    
        error_action="ignore",
        suppress_warnings=True,
        trace=True
    )
    sarima_models[area] = model


Fit SARIMA model for Pickup Community Area_0...

Best model:  ARIMA(1,1,3)(0,0,0)[0] intercept
Total fit time: 4.814 seconds
Fit SARIMA model for Pickup Community Area_1...

Best model:  ARIMA(1,1,2)(0,0,0)[0] intercept
Total fit time: 5.847 seconds
Fit SARIMA model for Pickup Community Area_2...

Best model:  ARIMA(2,1,2)(0,0,0)[0] intercept
Total fit time: 4.512 seconds
Fit SARIMA model for Pickup Community Area_3...

Best model:  ARIMA(3,1,2)(0,0,0)[0] intercept
Total fit time: 4.974 seconds
Fit SARIMA model for Pickup Community Area_4...

Best model:  ARIMA(1,1,1)(0,0,0)[0] intercept
Total fit time: 4.921 seconds
Fit SARIMA model for Pickup Community Area_5...

Best model:  ARIMA(1,1,2)(0,0,0)[0] intercept
Total fit time: 5.516 seconds
Fit SARIMA model for Pickup Community Area_6...

Best model:  ARIMA(2,1,3)(0,0,0)[0] intercept
Total fit time: 5.932 seconds
Fit SARIMA model for Pickup Community Area_7...

Best model:  ARIMA(2,1,2)(0,0,0)[0] intercept
Total fit time: 4.986 seconds


/home/isel-har/Documents/uber/.venv/lib/python3.11/site-packages/pmdarima/arima/auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '


 ARIMA(0,0,0)(0,0,0)[0]             : AIC=-136681.619, Time=0.24 sec
Total fit time: 0.238 seconds
Fit SARIMA model for Pickup Community Area_56...

Best model:  ARIMA(2,1,2)(0,0,0)[0] intercept
Total fit time: 5.635 seconds
Fit SARIMA model for Pickup Community Area_57...

Best model:  ARIMA(0,0,1)(0,0,0)[0]          
Total fit time: 1.425 seconds
Fit SARIMA model for Pickup Community Area_58...

Best model:  ARIMA(2,1,3)(0,0,0)[0] intercept
Total fit time: 10.003 seconds
Fit SARIMA model for Pickup Community Area_59...

Best model:  ARIMA(1,1,3)(0,0,0)[0] intercept
Total fit time: 3.483 seconds
Fit SARIMA model for Pickup Community Area_60...

Best model:  ARIMA(0,1,1)(0,0,0)[0] intercept
Total fit time: 8.371 seconds
Fit SARIMA model for Pickup Community Area_61...

Best model:  ARIMA(3,1,1)(0,0,0)[0] intercept
Total fit time: 8.435 seconds
Fit SARIMA model for Pickup Community Area_62...

Best model:  ARIMA(0,0,0)(0,0,0)[0]          
Total fit time: 1.577 seconds
Fit SARIMA model f

/home/isel-har/Documents/uber/.venv/lib/python3.11/site-packages/pmdarima/arima/auto.py:444: UserWarning: Input time-series is completely constant; returning a (0, 0, 0) ARMA.
  warnings.warn('Input time-series is completely constant; '


 ARIMA(0,0,0)(0,0,0)[0]             : AIC=-136681.619, Time=0.21 sec
Total fit time: 0.212 seconds
Fit SARIMA model for Pickup Community Area_75...

Best model:  ARIMA(0,1,1)(0,0,0)[0] intercept
Total fit time: 8.800 seconds
Fit SARIMA model for Pickup Community Area_76...

Best model:  ARIMA(2,1,2)(0,0,0)[0] intercept
Total fit time: 4.373 seconds
Fit SARIMA model for Pickup Community Area_77...

Best model:  ARIMA(3,1,2)(0,0,0)[0] intercept
Total fit time: 5.331 seconds
 ARIMA(2,1,1)(0,0,0)[0] intercept   : AIC=1885.108, Time=2.53 sec
 ARIMA(0,1,3)(0,0,0)[0] intercept   : AIC=1436.911, Time=2.27 sec
 ARIMA(1,1,1)(0,0,0)[0] intercept   : AIC=9140.305, Time=1.61 sec
 ARIMA(0,1,3)(0,0,0)[0] intercept   : AIC=8854.820, Time=2.18 sec
 ARIMA(1,1,3)(0,0,0)[0] intercept   : AIC=8840.248, Time=4.89 sec
 ARIMA(2,1,3)(0,0,0)[0] intercept   : AIC=7782.963, Time=4.83 sec
 ARIMA(2,1,3)(0,0,0)[0] intercept   : AIC=6615.410, Time=3.96 sec
 ARIMA(2,1,0)(0,0,0)[0] intercept   : AIC=7652.435, Time=0.58

compute the average of mean absolute error of best SARIMA Model for each Area

Forecast using best SARIMA models for each Area

In [21]:

sma_mae = []
sarima_forecasts = {}
for area in areas:
    forecast = sarima_models[area].predict(n_periods=len(val) + len(sub_df))
    forecast = np.expm1(forecast)
    sarima_forecasts[area] = forecast
    sma_mae.append(mean_absolute_error(forecast[:len(val)], val[area]))

mae_avg = np.array(sma_mae).mean()
print(f"Mean Absolute Error : {mae_avg}")

Mean Absolute Error : 0.8974713317877129


In [ ]:
Submit using ARIMA model

In [23]:
sub_pred = pd.DataFrame(columns=sub_df.columns)
sub_pred['Trip Start Timestamp'] = sub_df['Trip Start Timestamp']

for area in areas:

    forecast = sarima_forecasts[area][len(val):]
    sub_pred[area] = forecast.to_numpy()


sub_pred

,Trip Start Timestamp,Pickup Community Area_0,Pickup Community Area_1,Pickup Community Area_2,Pickup Community Area_3,Pickup Community Area_4,Pickup Community Area_5,Pickup Community Area_6,Pickup Community Area_7,Pickup Community Area_8,...,Pickup Community Area_68,Pickup Community Area_69,Pickup Community Area_70,Pickup Community Area_71,Pickup Community Area_72,Pickup Community Area_73,Pickup Community Area_74,Pickup Community Area_75,Pickup Community Area_76,Pickup Community Area_77
0,2019-06-24 00:00:00,5.859422,0.797973,0.027297,0.227984,0.062291,0.063829,-0.780979,0.694870,-0.400025,...,0.007227,-0.000864,-0.002688,-0.009304,-0.010063,0.007446,0.0,-0.015218,-0.408079,0.183048
1,2019-06-24 00:15:00,5.860911,0.798492,0.027297,0.227948,0.062295,0.063834,-0.781215,0.694968,-0.400608,...,0.007226,-0.000865,-0.002689,-0.009310,-0.010069,0.007442,0.0,-0.015227,-0.408194,0.183071
2,2019-06-24 00:30:00,5.862400,0.799012,0.027296,0.227911,0.062299,0.063840,-0.781452,0.695065,-0.401191,...,0.007225,-0.000865,-0.002691,-0.009316,-0.010075,0.007439,0.0,-0.015236,-0.408308,0.183095
3,2019-06-24 00:45:00,5.863890,0.799531,0.027296,0.227875,0.062303,0.063845,-0.781688,0.695163,-0.401773,...,0.007224,-0.000866,-0.002693,-0.009322,-0.010081,0.007436,0.0,-0.015245,-0.408423,0.183118
4,2019-06-24 01:00:00,5.865380,0.800050,0.027296,0.227839,0.062308,0.063850,-0.781924,0.695261,-0.402355,...,0.007223,-0.000866,-0.002694,-0.009327,-0.010086,0.007433,0.0,-0.015254,-0.408538,0.183141
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667,2019-06-30 22:45:00,6.928047,1.179648,0.027083,0.204087,0.064984,0.067275,-0.893514,0.761313,-0.686413,...,0.006556,-0.001221,-0.003755,-0.013129,-0.013965,0.005342,0.0,-0.021211,-0.479860,0.198658
668,2019-06-30 23:00:00,6.929768,1.180277,0.027083,0.204052,0.064988,0.067280,-0.893629,0.761415,-0.686718,...,0.006555,-0.001221,-0.003757,-0.013135,-0.013971,0.005339,0.0,-0.021220,-0.479961,0.198681
669,2019-06-30 23:15:00,6.931489,1.180907,0.027082,0.204016,0.064992,0.067286,-0.893744,0.761516,-0.687022,...,0.006554,-0.001222,-0.003758,-0.013141,-0.013977,0.005336,0.0,-0.021229,-0.480062,0.198705
670,2019-06-30 23:30:00,6.933211,1.181536,0.027082,0.203981,0.064996,0.067291,-0.893859,0.761618,-0.687327,...,0.006553,-0.001222,-0.003760,-0.013147,-0.013982,0.005333,0.0,-0.021238,-0.480162,0.198729
